# Domain 5: Context Management & Reliability
## TS 5.4: Manage Context in Large Codebase Exploration with Scratchpads & Human Review Routing
**Certification Focus**: CCAR-F Domain 5 (15% Exam Weight) | **Scenario Alignment**: Scenario 4 & 6

---

### 🎯 Key Architectural Principles
1. **The Working Memory Scratchpad**: During complex codebase explorations, keep raw file reads out of conversation history. Record findings into an external scratchpad (`CodebaseScratchpadManager`) and inject only the compact scratchpad summary into Claude.
2. **Confidence-Based Routing**:
   - Confidence >= 0.85: Auto-approve.
   - Confidence < 0.85: Route to human reviewers.
3. **Stratified Random Sampling**: To catch silent model drift or systemic inaccuracies, take a 5–10% random sample across **all document classes/segments** (even high-confidence items).

In [ ]:
# Step 1: Environment & Setup
import random
import sys
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 5.4 Scratchpads & Human Review Routing.")

### 📝 Step 2: Scratchpad & Human Review Routing Engine
We implement the working memory scratchpad and confidence router with stratified sampling.

In [ ]:
# Step 2: Scratchpad & Router Logic
@dataclass
class ExtractionRecord:
    doc_id: str
    category: str
    confidence: float
    extracted_amount: float
    requires_human_review: bool = False


class HumanReviewRoutingEngine:
    def __init__(self, confidence_threshold: float = 0.85) -> None:
        self.threshold = confidence_threshold

    def route_record(self, record: ExtractionRecord) -> ExtractionRecord:
        record.requires_human_review = (record.confidence < self.threshold)
        return record

    @staticmethod
    def stratified_random_sample(records: List[ExtractionRecord], sample_per_stratum: int = 1) -> List[ExtractionRecord]:
        """Selects representative audit samples across distinct categories/strata."""
        strata = defaultdict(list)
        for r in records:
            strata[r.category].append(r)
        
        sampled = []
        for cat, items in strata.items():
            sampled.extend(items[:sample_per_stratum])
        return sampled


class CodebaseScratchpadManager:
    """External working memory container preventing conversation context bloat."""
    def __init__(self) -> None:
        self.findings: List[str] = []

    def record_finding(self, observation: str) -> None:
        self.findings.append(observation)

    def render_scratchpad(self) -> str:
        return "### AGENT WORKING SCRATCHPAD\n" + "\n".join(f"- {f}" for f in self.findings)

print(" HumanReviewRoutingEngine and CodebaseScratchpadManager ready.")

### 🧪 Step 3: Interactive Simulation
1. Route high vs low confidence records.
2. Execute stratified audit sampling across document segments.
3. Record exploration observations into scratchpad.

In [ ]:
# Step 3: Simulation
router = HumanReviewRoutingEngine(confidence_threshold=0.85)

rec_high = router.route_record(ExtractionRecord("D1", "standard_invoice", 0.96, 120.0))
rec_low = router.route_record(ExtractionRecord("D2", "customs_declaration", 0.72, 450.0))

print("High Confidence (0.96) Needs Human:", rec_high.requires_human_review)
print("Low Confidence (0.72) Needs Human:", rec_low.requires_human_review)

# Stratified audit across categories
records = [
    ExtractionRecord("D10", "Category_A", 0.95, 10.0),
    ExtractionRecord("D11", "Category_A", 0.92, 20.0),
    ExtractionRecord("D20", "Category_B", 0.88, 30.0),
]
audit_sample = HumanReviewRoutingEngine.stratified_random_sample(records, sample_per_stratum=1)
print(f"\nStratified Sample Count: {len(audit_sample)} (1 from Category_A, 1 from Category_B)")

# Scratchpad memory test
pad = CodebaseScratchpadManager()
pad.record_finding("auth_service uses bcrypt with 12 rounds.")
pad.record_finding("redis cache TTL is set to 3600 seconds.")
print("\n", pad.render_scratchpad())

### 🔬 Step 4: Verification Assertions
We assert that:
1. High-confidence item bypasses human review.
2. Low-confidence item is routed to human review.
3. Stratified sampling captures 1 sample per distinct stratum.
4. Scratchpad renders clean, formatted findings.

In [ ]:
# Step 4: Verification Checks
assert rec_high.requires_human_review is False
assert rec_low.requires_human_review is True

assert len(audit_sample) == 2
strata_represented = {r.category for r in audit_sample}
assert "Category_A" in strata_represented
assert "Category_B" in strata_represented

scratchpad_text = pad.render_scratchpad()
assert "bcrypt" in scratchpad_text
assert "redis cache" in scratchpad_text

print(" All verification assertions passed! Scratchpads & review routing verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Maintain an **external scratchpad** for working memory during multi-step codebase exploration.
- **Rule 2**: Route items below the confidence threshold (<0.85) to human specialists.
- **Rule 3**: Apply **stratified random sampling** across document classes to audit high-confidence decisions for silent model drift.